# NB12 — Lasso Regression (L1 Regularisation)

> **L1 penalty drives coefficients to exactly zero — automatic feature selection.**

Lasso minimises: `SSR + λ·Σ|βⱼ|`

The L1 (absolute value) penalty has a sharp corner at zero.
The OLS solution often lands exactly at that corner → coefficient = 0 → feature dropped.

Ridge keeps all features (just small). Lasso selects features.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X_std = StandardScaler().fit_transform(housing.data)
y     = housing.target

lambdas = np.logspace(-4, 1, 200)
coefs   = []
for lam in lambdas:
    m = Lasso(alpha=lam, max_iter=10000).fit(X_std, y)
    coefs.append(m.coef_)
coefs = np.array(coefs)

plt.figure(figsize=(10, 5))
for j, name in enumerate(housing.feature_names):
    plt.semilogx(lambdas, coefs[:, j], label=name)
plt.xlabel('λ (log scale)'); plt.ylabel('Coefficient value')
plt.title('Lasso path — coefficients hit exactly zero (feature selection)')
plt.legend(fontsize=8, loc='upper right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# LassoCV — find optimal lambda by cross-validation
from sklearn.linear_model import LassoCV
import numpy as np

lasso_cv = LassoCV(cv=5, max_iter=20000, n_alphas=200).fit(X_std, y)
print(f"Optimal λ: {lasso_cv.alpha_:.6f}")
print(f"\nCoefficients at optimal λ:")
for name, coef in zip(housing.feature_names, lasso_cv.coef_):
    status = 'SELECTED' if coef != 0 else 'zeroed out'
    print(f"  {name:<15} {coef:>10.4f}   {status}")
n_selected = np.sum(lasso_cv.coef_ != 0)
print(f"\n{n_selected}/{len(housing.feature_names)} features selected")


## Why L1 produces sparsity — geometric intuition

The feasible region for L1 is a **diamond** (in 2D), which has corners on the axes.
The OLS objective (elliptical contours) tends to first touch the diamond at a corner → one coefficient = 0.

Ridge uses a **circle** — no corners → solution rarely lands on an axis.


In [ ]:
# Visualise the geometry in 2D
import numpy as np
import matplotlib.pyplot as plt

theta = np.linspace(0, 2*np.pi, 500)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# OLS contours (ellipse centred at the "true" OLS solution)
true_ols = np.array([1.5, 0.8])
for ax, title, boundary_fn, label in zip(axes,
    ['Ridge (L2): circle, no corners', 'Lasso (L1): diamond, corners on axes'],
    [lambda r: (r*np.cos(theta), r*np.sin(theta)),   # circle
     lambda r: None],
    ['L2 ball', 'L1 ball']):

    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal')
    ax.axhline(0, color='k', linewidth=0.5); ax.axvline(0, color='k', linewidth=0.5)

    # Constraint ball
    if 'Ridge' in title:
        ax.plot(np.cos(theta), np.sin(theta), 'b-', linewidth=2, label='L2 ball (r=1)')
    else:
        d = np.array([[1,0],[-1,0],[0,-1],[0,1],[1,0]])
        ax.plot(d[:,0], d[:,1], 'b-', linewidth=2, label='L1 ball (r=1)')

    # OLS contours
    for r in [0.4, 0.8, 1.2, 1.8]:
        cx = true_ols[0] + r*np.cos(theta)
        cy = true_ols[1] + 0.6*r*np.sin(theta)
        ax.plot(cx, cy, 'r-', alpha=0.4, linewidth=1)

    ax.plot(*true_ols, 'r*', markersize=14, label='OLS solution')
    # Approximate constrained solution
    if 'Lasso' in title:
        ax.plot(1, 0, 'go', markersize=12, label='Lasso solution (β₂=0!)')
    else:
        ax.plot(0.78, 0.62, 'go', markersize=12, label='Ridge solution')

    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Why L1 selects features: the diamond has corners on the axes', fontsize=11)
plt.tight_layout(); plt.show()


## Ridge vs Lasso: when to use which

| | Ridge | Lasso |
|--|-------|-------|
| Penalty | Σβⱼ² | Σ|βⱼ| |
| Variable selection | No | Yes |
| Sparse solution | No | Yes |
| When many features matter | Better | Worse |
| When few features matter | OK | Better |
| Stable with correlated features | Yes | Can be unstable |

**Next:** NB13 — ElasticNet combines both penalties.
